### Wtech SQL ve Veritabanı Eğitimi

In [1]:
%load_ext sql
%sql sqlite:///data/ecommerce.db

%config SqlMagic.displaylimit = None
%config SqlMagic.autolimit = 0

print("Bağlantı kuruldu: data/ecommerce.db")

Connecting to 'sqlite:///data/ecommerce.db'

displaylimit: Value None will be treated as 0 (no limit)

Bağlantı kuruldu: data/ecommerce.db


---

## SQL ile Veri Sorgulama

### SELECT, WHERE, ORDER BY

Müşteri tablosunda ne tür veriler var, yapısı nasıl — önce ilk birkaç satıra bakalım.

In [4]:
%%sql
select * from customers
limit 5;

Running query in 'sqlite:///data/ecommerce.db'

customer_id,first_name,last_name,email,phone,birth_date,gender,registration_date,city,customer_segment
1,Okyalaz,Bilgin,okyalaz.bilgin4690@gmail.com,+90+9 (509) 8393010,1965-05-28,erkek,2022-04-15,Bursa,silver
2,Amaç,Bilge,amac.bilge5773@gmail.com,+90(183)473 8299,1981-12-28,erkek,2023-01-13,Bursa,platinum
3,Bitül,Sezgin,bitul.sezgin4107@gmail.com,+90311 6 566,1981-08-09,belirtmek_istemiyorum,2022-03-26,Gaziantep,silver
4,Tanses,Şensoy,tanses.sensoy2241@gmail.com,+90+9 (065) 1333872,1978-07-14,belirtmek_istemiyorum,2022-10-13,İstanbul,silver
5,Sağcan,Ertaş,sagcan.ertas646@gmail.com,+901781080132,1977-06-25,kadin,2024-01-25,Gaziantep,platinum


Müşteri listesi çıkaracağız ama sadece ad, soyad, şehir ve segment bilgisi yeterli — gereksiz kolonları çekmeyelim.

In [6]:
%%sql
select first_name, city, customer_segment from customers
limit 5;

Running query in 'sqlite:///data/ecommerce.db'

first_name,city,customer_segment
Okyalaz,Bursa,silver
Amaç,Bursa,platinum
Bitül,Gaziantep,silver
Tanses,İstanbul,silver
Sağcan,Gaziantep,platinum


Sadece İstanbul, Ankara ve İzmir’deki müşterileri listeleyelim.

In [10]:
%%sql
select first_name, city, customer_segment 
from customers
--where city = 'Ankara' or city = 'İstanbul' or city = 'İzmir'
where city in ('Ankara', 'İstanbul', 'İzmir')
limit 10; 

Running query in 'sqlite:///data/ecommerce.db'

first_name,city,customer_segment
Tanses,İstanbul,silver
Deha,Ankara,gold
Gülağa,Ankara,silver
Demiryürek,İzmir,silver
Gülsalın,İstanbul,gold
Kaver,İzmir,silver
Verim,Ankara,silver
Güngören,İstanbul,silver
Alize,Ankara,silver
Orçin,Ankara,silver


2024’te teslim edilen siparişleri, en yüksek tutarlıdan başlayacak şekilde listeleyelim.

In [23]:
%%sql
select 
order_id, customer_id, order_date, total_amount 
from orders
where order_date between '2024-01-01' and '2024-12-31'
and order_status = 'teslim_edildi'
--and total_amount > 100
order by total_amount asc
limit 10;

Running query in 'sqlite:///data/ecommerce.db'

order_id,customer_id,order_date,total_amount
6741,3740,2024-05-03 20:54:21,20.22
2534,1379,2024-12-04 19:22:34,23.89
5519,3085,2024-07-31 20:36:06,36.0
7700,4273,2024-07-05 03:47:10,38.9
2994,1646,2024-12-27 00:42:40,40.5
7794,4335,2024-08-10 01:57:22,41.42
753,386,2024-02-15 10:57:42,45.0
1921,1017,2024-06-28 15:22:45,46.02
7007,3882,2024-07-01 07:47:56,48.63
8451,4719,2024-08-07 19:58:37,49.5


### GROUP BY ve Aggregation

Sipariş durumlarına göre kaç sipariş olduğunu öğrenelim — hazırlanan, kargodaki, teslim edilen, iptal edilen sayıları.

In [ ]:
%%sql
select order_status, count(*) as siparis_sayisi

from orders
where order_status in ('teslim_edildi', 'iptal', 'hazirlaniyor')
group by order_status
limit 5;

Running query in 'sqlite:///data/ecommerce.db'

order_status,siparis_sayisi
hazirlaniyor,1519
iptal,1466
teslim_edildi,4409


Ödeme yöntemlerine göre kaç sipariş verildi, toplam ne kadar ciro elde edildi, ortalama sepet tutarı ne — rapor hazırlayalım.

In [38]:
%%sql
select * from orders
limit 5;

Running query in 'sqlite:///data/ecommerce.db'

order_id,customer_id,order_date,order_status,total_amount,payment_method,shipping_cost,campaign_id
1,1,2023-05-31 20:04:04,kargoda,848.03,havale,34.99,None
2,1,2024-07-06 07:16:31,teslim_edildi,1886.68,kredi_karti,0.0,None
3,2,2025-06-06 19:07:17,teslim_edildi,8648.42,kapida_odeme,0.0,None
4,2,2023-04-14 05:40:22,teslim_edildi,2670.35,havale,0.0,3
5,2,2025-10-24 12:35:56,teslim_edildi,63478.89,kapida_odeme,24.99,None


In [43]:
%%sql
select payment_method, count(*) as siparis_sayisi,
round(sum(total_amount), 2) as toplam_ciro, 
round(avg(total_amount), 2) as ortalama_siparis_tutari

from orders 
group by payment_method
limit 5;

Running query in 'sqlite:///data/ecommerce.db'

payment_method,siparis_sayisi,toplam_ciro,ortalama_siparis_tutari
havale,2169,23080351.82,10641.01
kapida_odeme,2238,23607262.02,10548.37
kredi_karti,4516,47882517.44,10602.86


100’den fazla müşterisi olan şehirleri, müşteri sayısına göre sıralı listeleyelim.

In [65]:
%%sql

select city, count(*) as musteri_sayisi
from customers
group by city
having musteri_sayisi > 100
order by musteri_sayisi desc

Running query in 'sqlite:///data/ecommerce.db'

city,musteri_sayisi
İstanbul,1498
Ankara,749
İzmir,607
Bursa,356
Antalya,330
Konya,261
Adana,257
Gaziantep,237
Eskişehir,208
Mersin,186


### JOIN

In [69]:
%%sql
select * from orders limit 5;

Running query in 'sqlite:///data/ecommerce.db'

order_id,customer_id,order_date,order_status,total_amount,payment_method,shipping_cost,campaign_id
1,1,2023-05-31 20:04:04,kargoda,848.03,havale,34.99,None
2,1,2024-07-06 07:16:31,teslim_edildi,1886.68,kredi_karti,0.0,None
3,2,2025-06-06 19:07:17,teslim_edildi,8648.42,kapida_odeme,0.0,None
4,2,2023-04-14 05:40:22,teslim_edildi,2670.35,havale,0.0,3
5,2,2025-10-24 12:35:56,teslim_edildi,63478.89,kapida_odeme,24.99,None


In [68]:
%%sql
select * from customers limit 5;

Running query in 'sqlite:///data/ecommerce.db'

customer_id,first_name,last_name,email,phone,birth_date,gender,registration_date,city,customer_segment
1,Okyalaz,Bilgin,okyalaz.bilgin4690@gmail.com,+90+9 (509) 8393010,1965-05-28,erkek,2022-04-15,Bursa,silver
2,Amaç,Bilge,amac.bilge5773@gmail.com,+90(183)473 8299,1981-12-28,erkek,2023-01-13,Bursa,platinum
3,Bitül,Sezgin,bitul.sezgin4107@gmail.com,+90311 6 566,1981-08-09,belirtmek_istemiyorum,2022-03-26,Gaziantep,silver
4,Tanses,Şensoy,tanses.sensoy2241@gmail.com,+90+9 (065) 1333872,1978-07-14,belirtmek_istemiyorum,2022-10-13,İstanbul,silver
5,Sağcan,Ertaş,sagcan.ertas646@gmail.com,+901781080132,1977-06-25,kadin,2024-01-25,Gaziantep,platinum


In [76]:
%%sql
select * 
from customers c 
inner join orders o on o.customer_id = c.customer_id
limit 5;

Running query in 'sqlite:///data/ecommerce.db'

customer_id,first_name,last_name,email,phone,birth_date,gender,registration_date,city,customer_segment,order_id,customer_id_1,order_date,order_status,total_amount,payment_method,shipping_cost,campaign_id
1,Okyalaz,Bilgin,okyalaz.bilgin4690@gmail.com,+90+9 (509) 8393010,1965-05-28,erkek,2022-04-15,Bursa,silver,1,1,2023-05-31 20:04:04,kargoda,848.03,havale,34.99,None
1,Okyalaz,Bilgin,okyalaz.bilgin4690@gmail.com,+90+9 (509) 8393010,1965-05-28,erkek,2022-04-15,Bursa,silver,2,1,2024-07-06 07:16:31,teslim_edildi,1886.68,kredi_karti,0.0,None
2,Amaç,Bilge,amac.bilge5773@gmail.com,+90(183)473 8299,1981-12-28,erkek,2023-01-13,Bursa,platinum,3,2,2025-06-06 19:07:17,teslim_edildi,8648.42,kapida_odeme,0.0,None
2,Amaç,Bilge,amac.bilge5773@gmail.com,+90(183)473 8299,1981-12-28,erkek,2023-01-13,Bursa,platinum,4,2,2023-04-14 05:40:22,teslim_edildi,2670.35,havale,0.0,3
2,Amaç,Bilge,amac.bilge5773@gmail.com,+90(183)473 8299,1981-12-28,erkek,2023-01-13,Bursa,platinum,5,2,2025-10-24 12:35:56,teslim_edildi,63478.89,kapida_odeme,24.99,None


Her siparişin hangi şehirden geldiğini bulalım — sipariş tablosunda müşteri ID var, müşteri tablosunda şehir var; ikisini birleştireceğiz.

In [93]:
%%sql

select c.first_name, c.last_name, o.order_id, o.order_date, o.total_amount, c.city, cp.campaign_name
from orders o 
left join customers c on o.customer_id = c.customer_id
inner join campaigns cp on o.campaign_id = cp.campaign_id
where o.order_status = 'teslim_edildi'
and city = 'Diyarbakır'
and o.total_amount > 31012
order by o.total_amount desc

Running query in 'sqlite:///data/ecommerce.db'

first_name,last_name,order_id,order_date,total_amount,city,campaign_name
Okseven,Alemdar,5393,2024-03-11 13:08:06,31012.89,Diyarbakır,Bahar İndirimi


Tüm siparişleri listeleyelim; varsa kampanya adını da ekleyelim. Kampanyası olmayan siparişler de listede olsun.

Sipariş kalemlerinde hangi ürün var, hangi şehirden alındı — ürün adı, marka, miktar, fiyat ve müşteri şehrini birlikte görelim.

Ortalama puanı 3’ün altında kalan ve en az 10 yorumu olan markaları bulalım — ürün ekibi iyileştirme için öncelik verecek.

Siparişlere tutara göre segment atayalım — 10.000 TL üstü Yüksek Değer, 3.000–10.000 arası Orta Değer, altı Düşük Değer.

Her siparişin yanına, o müşterinin toplam harcamasını ekleyelim — müşteri bazında toplam görsün, satır sayısı aynı kalsın.

Ortalama puanı 3’ün altında kalan ve en az 10 yorumu olan markaları bulalım — ürün ekibi iyileştirme için öncelik verecek.

Her kategoride en çok satan ürünleri sıralayalım — kategori bazında 1., 2., 3. sıra gibi.

Ana kategorilere göre (Elektronik, Moda, Ev & Yaşam vb.) ne kadar ciro geldi, kaç sipariş verildi, ortalama sipariş tutarı ne — satış ekibine özet rapor.

Teslim edilen siparişlere göre hangi şehirler en çok ciro getirdi — ilk 5 şehri listeleyelim.